In [ ]:
import os
from pathlib import Path

import pandas as pd
from src import analysis
from src.analysis import evaluation

os.environ["CUDA_VISIBLE_DEVICES"] = ""  # Use CPU for evaluation

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
# Old Models trained on Dataset with scalar kappa
# Models trained on Dataset with scalar kappa
# modes(32, 32), hidden_channels=64
model = "FNO_lhs_var10_plog100_seed1_20251208_230304"
model = "FNO_lhs_var20_plog100_seed1001_20251210_163827"
model = "FNO_lhs_var40_plog100_seed2001_20251211_154034"
model = "FNO_lhs_var80_plog100_seed3001_20251212_125538"

# modes(12, 12), hidden_channels=24
model = "FNO_lhs_var80_plog100_seed3001_20260103_121739"

parts = model.split("_")
dataset_name_id = "_".join(parts[1:5])

dataset_names = [
    dataset_name_id,
    # "lhs_var20_plog100_seed1001",
    # "lhs_var40_plog100_seed2001",
    # "lhs_var80_plog100_seed3001",
    "lhs_var160_plog100_seed4001",
]

In [ ]:
# ======================================================================
# Models trained on Dataset with tensor kappa, porosity field
# and varying pressure boundary condition
# =====================================================================
# --- FNO ---
# FNO small
model = "FNO_m12x12_h24_l4_lhs_var80_seed3001_20260109_201346"
# FNO big

# --- PI-FNO ---
# PI-FNO small
model = "PI-FNO_m12x12_h24_l4_lamPhys1e-04_lamP5e-03_lhs_var80_seed3001_20260109_191704"
# PI-FNO big

# --- U-NO ---
# U-NO small

# U-NO big

# --- PI-U-NO ---
# PI-U-NO small

# PI-U-NO big

dataset_names = [
    "lhs_var80_seed3001",
    "lhs_var120_seed4001",
    # "lhs_var160_seed5001",
]

In [ ]:
# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------
checkpoint_path = Path(f"../data/processed/{model}/best_model_state_dict.pt")

# build cases paths automatically
dataset_cases_paths = {name: Path(f"../../data/raw/{name}/cases") for name in dataset_names}

# build save roots automatically
save_roots = {dataset_names[0]: Path(f"../data/processed/{model}/analysis/id")}

save_roots.update({name: Path(f"../data/processed/{model}/analysis/ood") / name for name in dataset_names[1:]})

In [ ]:
def run_or_load_artifacts_evaluation(
    dataset_name: str,
    save_root: Path,
    dataset_path: Path,
) -> tuple[pd.DataFrame, Path]:
    """Load an existing Parquet artifact for the given dataset, or create it if missing."""
    parquet_path = save_root / f"{dataset_name}.parquet"

    if parquet_path.exists():
        print(f"[INFO] Found existing parquet -> loading: {parquet_path}")
        df = pd.read_parquet(parquet_path)
        print(f"[INFO] Loaded df with {len(df)} rows")
        return df, parquet_path

    print(f"[INFO] Creating artifacts for: {dataset_name}")

    model, loader, processor, device = analysis.analysis_interference.load_inference_context(
        dataset_path=dataset_path,
        checkpoint_path=checkpoint_path,
        batch_size=1,
    )

    df, parquet_path = analysis.analysis_artifacts.generate_artifacts(
        model=model,
        loader=loader,
        processor=processor,
        device=device,
        save_root=save_root,
        dataset_name=dataset_name,
    )

    print("[INFO] Artifact generation done.")
    return df, parquet_path

In [ ]:
# ---------------------------------------------------------------------
# Generate artifacts
# ---------------------------------------------------------------------
datasets_raw = {}
parquet_paths = {}

for name in dataset_names:
    df_raw, parquet_path = run_or_load_artifacts_evaluation(
        dataset_name=name,
        save_root=save_roots[name],
        dataset_path=dataset_cases_paths[name],
    )
    datasets_raw[name] = df_raw
    parquet_paths[name] = parquet_path

In [ ]:
datasets_eval = {name: evaluation.evaluation_dataframe.load_and_build_eval_df(parquet_paths[name]) for name in dataset_names}

In [ ]:
panel = evaluation.evaluation_panel.build_evaluation_panel(
    datasets_eval=datasets_eval,
    title=model,
    sections=[
        "global_error",
        "error_decomposition",
        "physical_consistency",
        "spectral",
        "error_sensitivity",
        "sample_viewer",
        "outliers",
    ],
)

display(panel)